In [15]:
import pandas as pd
import numpy as np

from catboost import CatBoostRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

import warnings
warnings.filterwarnings("ignore")

In [16]:
master = pd.read_excel(r"C:\Users\Amey\Desktop\Amey\Python\100\Preprocessing\100_Pre_done_Combined.xlsx")

locations = pd.read_csv(
    r"C:\Users\Amey\Desktop\Amey\Python\locations.csv"
)

print(master.shape)
print(locations.shape)

(21349, 17)
(87, 4)


In [17]:
master['SLoc'] = (
    master['SLoc']
    .astype(str)
    .str.strip()
)

locations['SLOC'] = (
    locations['SLOC']
    .astype(str)
    .str.strip()
)

In [18]:
master = master.merge(

    locations[
        [
            'SLOC',
            'Latitude',
            'Longitude'
        ]
    ],

    left_on='SLoc',
    right_on='SLOC',

    how='left'
)

master.head()

,Material,SLoc,Quantity,Pstng Date,order,Equipment,Technician name,Year,Tavg,Tmax,Tmin,RH,Month,Season,Delta_T,Region,Location,SLOC,Latitude,Longitude
0,100,5023,-1,2020-01-04,48550533,10930429,Anil Sharma,2020,12.94,21.27,7.26,58.52,1,Winter,14.01,North1,Haryana,5023,29.000000,76.000000
1,100,5024,-1,2020-01-06,48556766,10844557,Jogendra Singh,2020,14.94,22.58,9.03,56.47,1,Winter,13.55,North1,Jaipur,5024,26.915458,75.818982
2,100,5030,-1,2020-01-06,48550093,10517828,Himanshu Kushwaha,2020,13.70,21.92,8.25,55.23,1,Winter,13.67,North1,Delhi,5030,28.613895,77.209006
3,100,5002,-1,2020-01-06,48550185,10519283,Prasad Chokhat,2020,22.83,30.49,16.34,68.83,1,Winter,14.15,West1,Navi Mumbai,5002,19.030826,73.019854
4,100,5044,-1,2020-01-06,48554032,10836205,Joby Varghese,2020,28.71,32.68,25.08,68.80,1,Winter,7.60,South,Cochin,5044,9.967903,76.244438


In [19]:
missing = master[
    master['Latitude'].isna()
]

print(
    "Missing Coordinates:",
    missing['SLoc'].nunique()
)

Missing Coordinates: 0


In [22]:
master['Pstng Date'] = pd.to_datetime(
    master['Pstng Date'],
    errors='coerce'
)

master = master.dropna(
    subset=['Pstng Date']
)

print(master['Pstng Date'].dtype)

datetime64[us]


In [23]:
monthly = (
    master
    .groupby(
        [
            'Year',
            'Month',
            'SLoc'
        ]
    )
    .agg(
        Consumption=('Quantity', lambda x: abs(x.sum())),
        Tavg=('Tavg','mean'),
        RH=('RH','mean'),
        Delta_T=('Delta_T','mean'),
        Region=('Region','first'),
        Location=('Location','first'),
        Season=('Season','first'),
        Latitude=('Latitude','first'),
        Longitude=('Longitude','first')
    )
    .reset_index()
)

monthly['Pstng Date'] = pd.to_datetime(
    monthly['Year'].astype(str)
    + '-'
    + monthly['Month'].astype(str)
    + '-01'
)

monthly.head()

,Year,Month,SLoc,Consumption,Tavg,RH,Delta_T,Region,Location,Season,Latitude,Longitude,Pstng Date
0,2020,1,5001,19,21.661579,67.084211,13.224737,West1,Mumbai,Winter,19.054999,72.869203,2020-01-01
1,2020,1,5002,4,22.910000,67.287500,11.530000,West1,Navi Mumbai,Winter,19.030826,73.019854,2020-01-01
2,2020,1,5005,1,18.910000,54.610000,12.240000,West2,Ahmedabad,Winter,23.021537,72.580057,2020-01-01
3,2020,1,5007,1,18.770000,74.430000,13.940000,West2,Raipur,Winter,21.238091,81.633699,2020-01-01
4,2020,1,5010,2,21.490000,63.915000,14.455000,West2,Surat,Winter,21.209489,72.831706,2020-01-01


In [24]:
monthly['sin_month'] = np.sin(
    2*np.pi*monthly['Month']/12
)

monthly['cos_month'] = np.cos(
    2*np.pi*monthly['Month']/12
)

In [25]:
monthly = monthly.sort_values(
    [
        'SLoc',
        'Pstng Date'
    ]
)

monthly.head()

,Year,Month,SLoc,Consumption,Tavg,RH,Delta_T,Region,Location,Season,Latitude,Longitude,Pstng Date,sin_month,cos_month
0,2020,1,5001,19,21.661579,67.084211,13.224737,West1,Mumbai,Winter,19.054999,72.869203,2020-01-01,5.000000e-01,8.660254e-01
16,2020,2,5001,21,26.452381,45.686667,15.245238,West1,Mumbai,Winter,19.054999,72.869203,2020-02-01,8.660254e-01,5.000000e-01
32,2020,3,5001,15,25.628667,52.732000,14.187333,West1,Mumbai,Summer,19.054999,72.869203,2020-03-01,1.000000e+00,6.123234e-17
51,2020,5,5001,5,32.292000,57.042000,13.442000,West1,Mumbai,Summer,19.054999,72.869203,2020-05-01,5.000000e-01,-8.660254e-01
59,2020,6,5001,21,28.118261,85.640435,5.165652,West1,Mumbai,Summer,19.054999,72.869203,2020-06-01,1.224647e-16,-1.000000e+00


In [26]:
for lag in [1,2,3,6,12]:

    monthly[f'lag_{lag}'] = (
        monthly
        .groupby('SLoc')['Consumption']
        .shift(lag)
    )

In [27]:
monthly['roll_3'] = (
    monthly
    .groupby('SLoc')['Consumption']
    .transform(
        lambda x:
        x.shift(1)
         .rolling(3)
         .mean()
    )
)

monthly['roll_6'] = (
    monthly
    .groupby('SLoc')['Consumption']
    .transform(
        lambda x:
        x.shift(1)
         .rolling(6)
         .mean()
    )
)

In [28]:
monthly = (
    monthly
    .dropna()
    .reset_index(drop=True)
)

print(monthly.shape)

(1164, 22)


In [29]:
weather_climatology = (
    monthly
    .groupby(
        [
            'Location',
            'Month'
        ]
    )
    .agg(
        Tavg=('Tavg','mean'),
        RH=('RH','mean'),
        Delta_T=('Delta_T','mean')
    )
    .reset_index()
)

weather_climatology.head()

,Location,Month,Tavg,RH,Delta_T
0,Ahmedabad,1,19.261333,40.136333,17.916000
1,Ahmedabad,2,23.855179,33.672054,17.825982
2,Ahmedabad,3,28.310345,27.296857,18.016357
3,Ahmedabad,4,33.503367,27.133300,17.913133
4,Ahmedabad,5,34.235833,38.597521,14.531937


In [30]:
split_date = '2025-01-01'

train = monthly[
    monthly['Pstng Date']
    < split_date
]

val = monthly[
    monthly['Pstng Date']
    >= split_date
]

print(train.shape)
print(val.shape)

(825, 22)
(339, 22)


In [31]:
features = [

    'SLoc',
    'Region',
    'Location',
    'Season',

    'Latitude',
    'Longitude',

    'Tavg',
    'RH',
    'Delta_T',

    'Month',
    'Year',

    'sin_month',
    'cos_month',

    'lag_1',
    'lag_2',
    'lag_3',
    'lag_6',
    'lag_12',

    'roll_3',
    'roll_6'
]

target = 'Consumption'

In [32]:
from catboost import CatBoostRegressor
cat_features = [

    'SLoc',
    'Region',
    'Location',
    'Season'
]

model = CatBoostRegressor(

    iterations=1000,
    learning_rate=0.03,
    depth=8,

    loss_function='RMSE',

    verbose=100
)

model.fit(

    train[features],
    train[target],

    cat_features=cat_features
)

0:	learn: 23.3247052	total: 63.2ms	remaining: 1m 3s
100:	learn: 6.9243411	total: 6.44s	remaining: 57.3s
200:	learn: 4.9012270	total: 12.5s	remaining: 49.5s
300:	learn: 4.0156537	total: 18.8s	remaining: 43.6s
400:	learn: 3.3953033	total: 25s	remaining: 37.3s
500:	learn: 2.9630233	total: 31.7s	remaining: 31.6s
600:	learn: 2.5879140	total: 38.1s	remaining: 25.3s
700:	learn: 2.2461727	total: 44.6s	remaining: 19s
800:	learn: 2.0022009	total: 50.9s	remaining: 12.7s
900:	learn: 1.7752079	total: 57s	remaining: 6.27s
999:	learn: 1.5941104	total: 1m 3s	remaining: 0us


CatBoostRegressor(depth=8, iterations=1000, learning_rate=0.03, loss_function='RMSE', verbose=100)

In [36]:
val_pred = model.predict(
    val[features]
)

mae = mean_absolute_error(
    val[target],
    val_pred
)

rmse = np.sqrt(
    mean_squared_error(
        val[target],
        val_pred
    )
)

# Remove zero actuals for MAPE

mask = val[target] != 0

mape = np.mean(
    np.abs(
        (
            val.loc[mask, target]
            - val_pred[mask]
        )
        /
        val.loc[mask, target]
    )
) * 100

print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"MAPE : {mape:.2f}%")
from sklearn.metrics import r2_score
r2 = r2_score(
    val[target],
    val_pred
)

print(f"R²   : {r2:.3f}")

MAE  : 5.32
RMSE : 11.64
MAPE : 62.03%
R²   : 0.887
